# POI Category × Taxi Volume Analysis

Examining how different POI categories relate to yellow taxi pickup volume across NYC zones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import ipywidgets as widgets
from IPython.display import display

POI_COLS = [c for c in pd.read_parquet("data/processed/final/zone_date_master_2025.parquet").columns
            if c.startswith("poi_count_") and c != "poi_count_total"]
TARGET = "trips_pickup_yellow"

label_map = {c: c.replace("poi_count_", "").replace("_", " ").title() for c in POI_COLS}
label_map[TARGET] = "Taxi Pickups (mean/day)"

## Interactive: POI × Taxi Volume by Year (2019 / 2020 / 2025)

Select a year to see how the correlation pattern shifts across time.

In [ ]:
import io
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Image
import matplotlib.pyplot as plt
from scipy import stats

# ── aggregate by zone ─────────────────────────────────────────────────────────
dfs = {}
for year in [2019, 2020, 2025]:
    raw = pd.read_parquet(f"data/processed/final/zone_date_master_{year}.parquet")
    zone_yr = (
        raw.groupby("taxi_zone_id")
        .agg({**{c: "first" for c in POI_COLS}, TARGET: "mean"})
        .dropna(subset=POI_COLS + [TARGET])
        .reset_index()
    )
    dfs[year] = zone_yr
    print(f"{year}: {len(zone_yr)} zones")

# ── pre-compute per-year Pearson r ────────────────────────────────────────────
def compute_corrs(year):
    z = dfs[year]
    rows = []
    for col in POI_COLS:
        r, p = stats.pearsonr(z[col], z[TARGET])
        rows.append({"col": col, "category": label_map[col], "r": r, "p": p, "sig": p < 0.05})
    return pd.DataFrame(rows)

corr_dfs = {y: compute_corrs(y) for y in [2019, 2020, 2025]}

# ── plot function ─────────────────────────────────────────────────────────────
out = widgets.Output()

def plot_year(year):
    plt.close("all")
    cd = corr_dfs[year].sort_values("r", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 7))

    bar_colors = ["#d73027" if r < 0 else "#1a9850" for r in cd["r"]]
    bars = ax.barh(cd["category"], cd["r"], color=bar_colors, edgecolor="white", height=0.6)

    # Annotate r values and significance stars
    for bar, r_val, sig in zip(bars, cd["r"], cd["sig"]):
        x = bar.get_width()
        label = f"{r_val:.2f}{'★' if sig else ''}"
        ax.text(
            x + (0.02 if x >= 0 else -0.02),
            bar.get_y() + bar.get_height() / 2,
            label, va="center",
            ha="left" if x >= 0 else "right",
            fontsize=9, color="black"
        )

    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("Pearson r  (correlation with yellow taxi pickups)", fontsize=11)
    ax.set_title(f"POI Category → Taxi Pickup Correlation — {year}\n(★ p < 0.05,  n ≈ 260 zones)",
                 fontsize=13, fontweight="bold")
    ax.set_xlim(-0.55, 0.80)
    ax.tick_params(labelsize=10)
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    fig.savefig(f"outputs/corr_poi_taxi_{year}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    buf.seek(0)
    with out:
        out.clear_output(wait=True)
        display(Image(buf.read()))

# ── widget ────────────────────────────────────────────────────────────────────
year_selector = widgets.ToggleButtons(
    options=[2019, 2020, 2025],
    description="Year:",
    button_style="info",
    style={"button_width": "80px"},
)
year_selector.observe(lambda change: plot_year(change["new"]), names="value")
display(widgets.VBox([year_selector, out]))
plot_year(2019)


## All-Years Comparison: POI → Taxi Correlation

Side-by-side grouped bars for all three years — makes the stability (or disruption) of each POI category's relationship immediately visible.

In [ ]:
import io

palette = {2019: "#2166ac", 2020: "#d6604d", 2025: "#4dac26"}
x     = np.arange(len(POI_COLS))
width = 0.25

fig, ax = plt.subplots(figsize=(15, 6))
for i, year in enumerate([2019, 2020, 2025]):
    cd_ordered = corr_dfs[year].set_index("col").loc[POI_COLS]
    ax.bar(x + i * width, cd_ordered["r"], width,
           label=str(year), color=palette[year], edgecolor="white", alpha=0.88)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x + width)
ax.set_xticklabels([label_map[c] for c in POI_COLS], rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Pearson r with Yellow Taxi Pickups")
ax.set_title("POI → Taxi Correlation Across Years (2019 / 2020 / 2025)",
             fontsize=13, fontweight="bold")
ax.legend(title="Year", fontsize=10)
ax.set_ylim(-0.45, 0.80)
plt.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
fig.savefig("outputs/corr_poi_taxi_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))
print("Saved → outputs/corr_poi_taxi_comparison.png")

## 1. Log-Scale Scatter: Each POI Category vs Taxi Pickups

Since taxi pickup counts are highly right-skewed, we use **log(1 + pickups)** as the y-axis. Each point is one taxi zone; the red line is the OLS fit. ★ marks statistically significant correlations (p < 0.05).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import io
from IPython.display import display, Image

# 合并三年数据
z_all = pd.concat(dfs.values(), ignore_index=True)
z_all["taxi_log"] = np.log1p(z_all[TARGET])

n_cats = len(POI_COLS)
ncols = 4
nrows = (n_cats + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(POI_COLS):
    ax = axes[i]
    x = z_all[col].values
    y = z_all["taxi_log"].values

    ax.scatter(x, y, alpha=0.25, s=12, color="#2166ac", edgecolors="none")

    r, p = stats.pearsonr(x, y)
    slope, intercept, *_ = stats.linregress(x, y)
    xline = np.linspace(x.min(), x.max(), 200)
    ax.plot(xline, intercept + slope * xline, color="#d73027", lw=1.8)

    sig_star = "★" if p < 0.05 else ""
    ax.set_title(f"{label_map[col]}\nr = {r:.2f}  p = {p:.3f} {sig_star}", fontsize=9)
    ax.set_xlabel("POI count (zone total)", fontsize=8)
    ax.set_ylabel("log(1 + taxi pickups/day)", fontsize=8)
    ax.tick_params(labelsize=7)

for j in range(n_cats, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("POI Count vs log(Taxi Pickups) — Per Category, Zone-Level (2019 + 2020 + 2025 combined)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
fig.savefig("outputs/scatter_poi_taxi.png", dpi=150, bbox_inches="tight")
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))
print("Saved → outputs/scatter_poi_taxi.png")


## 2. Multi-Feature Regression: How Well Does POI Composition Predict Taxi Volume?

Correlation is one-vs-one. Here we ask: **combined**, how much of the zone-level variation in taxi pickups is explained by all 13 POI features?  
We fit a Ridge regression (standardized features → log taxi pickups) per year and compare R² values, plus a coefficient heatmap to see which features the model leans on.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
from IPython.display import display, Image
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# ── Fit Ridge regression per year: all 13 POI features → log(taxi pickups) ───
reg_results = {}
for year in [2019, 2020, 2025]:
    z = dfs[year].copy()
    X = z[POI_COLS].values.astype(float)
    y = np.log1p(z[TARGET].values)

    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    model = Ridge(alpha=1.0)
    model.fit(X_s, y)

    reg_results[year] = {
        "r2": r2_score(y, model.predict(X_s)),
        "coefs": dict(zip(POI_COLS, model.coef_)),
    }

# ── Figure: R² bars + coefficient heatmap ────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: R² per year
years_list = [2019, 2020, 2025]
r2_vals = [reg_results[y]["r2"] for y in years_list]
bar_colors = ["#2166ac", "#d6604d", "#4dac26"]
bars = ax1.bar([str(y) for y in years_list], r2_vals, color=bar_colors, width=0.5)
for bar, val in zip(bars, r2_vals):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f"{val:.3f}", ha="center", va="bottom", fontsize=12, fontweight="bold")
ax1.set_ylim(0, 1.0)
ax1.set_ylabel("R²  (log-scaled target)", fontsize=11)
ax1.set_title("Ridge Regression R²\n(All 13 POI features → log Taxi Pickups)", fontsize=11)
ax1.axhline(0.5, color="gray", ls="--", lw=0.8, label="R²=0.5 reference")
ax1.legend(fontsize=9)

# Panel 2: Coefficient heatmap
coef_df = pd.DataFrame(
    {str(y): [reg_results[y]["coefs"][c] for c in POI_COLS] for y in years_list},
    index=[label_map[c] for c in POI_COLS],
)
sns.heatmap(coef_df, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            linewidths=0.5, ax=ax2, annot_kws={"size": 9})
ax2.set_title("Standardized Coefficients per Year\n(positive = more POI → more pickups)", fontsize=11)
ax2.set_xlabel("Year")
ax2.tick_params(labelsize=9)

plt.suptitle("How Well Do POI Features Collectively Explain Taxi Volume?",
             fontsize=13, fontweight="bold")
plt.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
fig.savefig("outputs/regression_poi_taxi.png", dpi=150, bbox_inches="tight")
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))
print("Saved → outputs/regression_poi_taxi.png")


## 3. Zone Archetypes: Clustering by POI Mix

Rather than looking at one category at a time, we cluster all zones by their **normalized POI composition** (K-Means, K=5). Each cluster is a zone "archetype" — e.g., commercial-heavy, residential, transit-oriented — and we compare how taxi volumes differ across archetypes.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io
from IPython.display import display, Image
from sklearn.cluster import KMeans

year = 2025
z = dfs[year].copy()

# Normalize each zone's POI counts to proportions (POI mix profile)
X_raw = z[POI_COLS].values.astype(float)
row_sums = X_raw.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
X_norm = X_raw / row_sums

# K-Means: 5 zone archetypes
N_CLUSTERS = 5
km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
z["cluster"] = km.fit_predict(X_norm)

# Label clusters: track duplicates and append 1/2
raw_labels = {}
for c in range(N_CLUSTERS):
    dominant_idx = np.argmax(km.cluster_centers_[c])
    raw_labels[c] = label_map[POI_COLS[dominant_idx]]

name_count = {}
cluster_labels = {}
for c in range(N_CLUSTERS):
    name = raw_labels[c]
    name_count[name] = name_count.get(name, 0) + 1

seen = {}
for c in range(N_CLUSTERS):
    name = raw_labels[c]
    if name_count[name] > 1:
        seen[name] = seen.get(name, 0) + 1
        cluster_labels[c] = f"C{c}: {name} {seen[name]}"
    else:
        cluster_labels[c] = f"C{c}: {name}"

z["cluster_name"] = z["cluster"].map(cluster_labels)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Panel 1 – POI profile heatmap per cluster
center_df = pd.DataFrame(
    km.cluster_centers_,
    columns=[label_map[c] for c in POI_COLS],
    index=[cluster_labels[i] for i in range(N_CLUSTERS)],
)
sns.heatmap(center_df, annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.5, ax=ax1, annot_kws={"size": 8})
ax1.set_title("POI Mix Profile per Zone Archetype\n(normalized proportions)", fontsize=11)
ax1.tick_params(axis="x", rotation=40, labelsize=8)
ax1.tick_params(axis="y", labelsize=9)

# Panel 2 – Box plot of daily taxi pickups per cluster
order = z.groupby("cluster_name")[TARGET].median().sort_values(ascending=False).index.tolist()
cluster_data = [z[z["cluster_name"] == name][TARGET].values for name in order]
bp = ax2.boxplot(cluster_data, vert=True, patch_artist=True,
                 showfliers=False, medianprops=dict(color="black", lw=2))
colors_box = plt.cm.Set2(np.linspace(0, 1, N_CLUSTERS))
for patch, color in zip(bp["boxes"], colors_box):
    patch.set_facecolor(color)
ax2.set_xticklabels(order, rotation=20, ha="right", fontsize=9)
ax2.set_ylabel("Daily Taxi Pickups (mean per zone)", fontsize=10)
ax2.set_title(f"Taxi Volume Distribution per Zone Archetype ({year})", fontsize=11)

plt.suptitle("Zone Archetypes from POI Mix Clustering (K=5)", fontsize=13, fontweight="bold")
plt.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
fig.savefig("outputs/cluster_poi_taxi.png", dpi=150, bbox_inches="tight")
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))

summary = (
    z.groupby("cluster_name")[TARGET]
    .agg(n_zones="count", median_daily_pickups="median", mean_daily_pickups="mean")
    .loc[order]
    .round(1)
)
print(summary)
print("\nSaved → outputs/cluster_poi_taxi.png")


## 4. Weekday vs Weekend: Does the POI–Taxi Relationship Shift?

Splitting days into weekday / weekend reveals whether certain POI types (e.g. recreational, cultural) are more predictive of taxi demand on weekends, while commercial / government drive weekday demand.

In [ ]:

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import io
from IPython.display import display, Image

# 合并三年数据
raw_all = pd.concat([
    pd.read_parquet(f"data/processed/final/zone_date_master_{year}.parquet")
    for year in [2019, 2020, 2025]
], ignore_index=True)
raw_all["date"] = pd.to_datetime(raw_all["date"])
raw_all["is_weekend"] = raw_all["date"].dt.weekday >= 5

corrs = {}
for label, flag in [("Weekday", False), ("Weekend", True)]:
    sub = raw_all[raw_all["is_weekend"] == flag]
    zone_sub = (
        sub.groupby("taxi_zone_id")
        .agg({**{c: "first" for c in POI_COLS}, TARGET: "mean"})
        .dropna(subset=POI_COLS + [TARGET])
    )
    corrs[label] = {col: stats.pearsonr(zone_sub[col], zone_sub[TARGET])[0] for col in POI_COLS}

diff = (pd.Series(corrs["Weekend"]) - pd.Series(corrs["Weekday"])).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#1a9850" if v >= 0 else "#d73027" for v in diff]
ax.barh([label_map[c] for c in diff.index], diff.values, color=colors, edgecolor="white", height=0.6)
ax.axvline(0, color="black", linewidth=0.9)
ax.set_xlabel("Weekend r − Weekday r", fontsize=11)
ax.set_title("Which POI Types Matter More on Weekends vs Weekdays?\n(2019 + 2020 + 2025 combined,  green = weekend stronger)",
             fontsize=12, fontweight="bold")
ax.tick_params(labelsize=10)
plt.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
fig.savefig("outputs/weekday_weekend_poi_taxi.png", dpi=150, bbox_inches="tight")
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))
print("Saved → outputs/weekday_weekend_poi_taxi.png")


## 5. Spatial Map: Taxi Volume vs POI Density

Choropleth maps showing where taxi pickups are high and where key POI categories concentrate — the spatial overlap directly illustrates the correlations we found above.

In [ ]:

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm
import ipywidgets as widgets
from IPython.display import display

# ── Prepare zone-level data ───────────────────────────────────────────────────
zones_gdf = gpd.read_file("data/raw/taxi_zones/taxi_zones.shp")
zones_gdf = zones_gdf.rename(columns={"LocationID": "taxi_zone_id"})
zones_gdf["taxi_zone_id"] = zones_gdf["taxi_zone_id"].astype(int)

z_all = pd.concat(dfs.values(), ignore_index=True)
zone_agg = (
    z_all.groupby("taxi_zone_id")
    .agg({TARGET: "mean", **{c: "first" for c in POI_COLS}})
    .reset_index()
)
zone_agg["taxi_zone_id"] = zone_agg["taxi_zone_id"].astype(int)
zone_agg["taxi_log"] = np.log1p(zone_agg[TARGET])
for c in POI_COLS:
    zone_agg[c + "_log"] = np.log1p(zone_agg[c])

gdf = (
    zones_gdf[["taxi_zone_id", "zone", "borough", "geometry"]]
    .merge(zone_agg, on="taxi_zone_id", how="left")
    .to_crs("EPSG:4326")
)

# ── Dropdown options ──────────────────────────────────────────────────────────
MAP_OPTIONS = {
    "Daily Taxi Pickups (avg)": ("taxi_log", TARGET, ["#ffffcc", "#fd8d3c", "#bd0026"], "log(1 + daily pickups)"),
    **{
        label_map[c]: (c + "_log", c, ["#eff3ff", "#6baed6", "#084594"], f"log(1 + {label_map[c]} POI count)")
        for c in POI_COLS
    },
}

# ── Map builder ───────────────────────────────────────────────────────────────
def make_map(option_name):
    log_col, raw_col, colors, legend = MAP_OPTIONS[option_name]

    vmin = gdf[log_col].quantile(0.05)
    vmax = gdf[log_col].quantile(0.95)
    colormap = cm.LinearColormap(colors, vmin=vmin, vmax=vmax, caption=legend)

    m = folium.Map(location=[40.7128, -74.0060], zoom_start=10, tiles="cartodbpositron")

    def style_fn(feature):
        val = feature["properties"].get(log_col)
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return {"fillColor": "#cccccc", "color": "white", "weight": 0.5, "fillOpacity": 0.5}
        return {"fillColor": colormap(val), "color": "white", "weight": 0.5, "fillOpacity": 0.75}

    tooltip_fields = ["zone", "borough", TARGET]
    tooltip_aliases = ["Zone:", "Borough:", "Avg Daily Pickups:"]
    if raw_col != TARGET:
        tooltip_fields.append(raw_col)
        tooltip_aliases.append(f"{option_name}:")

    folium.GeoJson(
        gdf,
        style_function=style_fn,
        tooltip=folium.GeoJsonTooltip(
            fields=tooltip_fields,
            aliases=tooltip_aliases,
            localize=True,
            sticky=True,
        ),
    ).add_to(m)
    colormap.add_to(m)
    return m

# ── Widget ────────────────────────────────────────────────────────────────────
out_map = widgets.Output()

dropdown = widgets.Dropdown(
    options=list(MAP_OPTIONS.keys()),
    value="Daily Taxi Pickups (avg)",
    description="Show:",
    layout=widgets.Layout(width="380px"),
    style={"description_width": "60px"},
)

def on_change(change):
    with out_map:
        out_map.clear_output(wait=True)
        m = make_map(change["new"])
        m.save("outputs/interactive_poi_map.html")
        display(m)

dropdown.observe(on_change, names="value")
display(widgets.VBox([dropdown, out_map]))
on_change({"new": dropdown.value})
